# Kumo-Tabular vs. tuned boosting and LDA

**Question:** can a pretrained tabular foundation model, used as-is, compete with tuned classical models, with and without domain-inspired feature engineering?

| Task | Datasets | Models |
|---|---|---|
| Classification | Iris, Wine, Breast Cancer | Kumo, Gradient Boosting, Histogram Gradient Boosting, XGBoost, CatBoost, standardized LDA |
| Regression | Diabetes, Air Quality (ozone), Auto MPG | Kumo, Gradient Boosting, Histogram Gradient Boosting, XGBoost, CatBoost |

**Protocol**

- One fixed 75/25 train/test split (seed 42; stratified for classification). Raw and engineered scenarios use the same split.
- **Tuning:** each non-Kumo model is tuned with Optuna (TPE sampler, `N_TRIALS` trials), using five-fold cross-validation on the training split only. Classification optimizes log loss; regression optimizes RMSE. The best setting is refit on the full training split and evaluated once on the test data.
- **Kumo** uses pretrained weights without dataset-specific weight updates or hyperparameter search. The labeled training rows are its in-context context. Its default preprocessing ensemble (`NUM_ESTIMATORS`) is used. Regression point predictions use the median quantile, `q500`.
- **Two paired scenarios:** `Raw` uses original features; `Feature engineered` adds the row-wise features listed below. No statistics are learned from the test rows. Both scenarios use the same split.

**Feature engineering**

| Dataset | Added features |
|---|---|
| Iris | petal and sepal area, petal aspect ratio, petal-to-sepal length ratio |
| Wine | flavanoid share of total phenols, colour intensity / hue, log proline, flavanoids × OD280/OD315 |
| Breast Cancer | worst/mean ratios for radius, texture and concave points; perimeter²/area; worst area × worst concave points |
| Diabetes | BMI × blood pressure, BMI × s5, blood pressure × s5, LDL/HDL (s2/s3), BMI² |
| Air Quality | day of year, temperature / wind, solar radiation × temperature, 1 / wind |
| Auto MPG | horsepower / weight, displacement per cylinder, log weight, one-hot origin (replaces the numeric code) |

**Caveats:** this is a small, single-split benchmark, not a definitive leaderboard. Test sets are small, so a few predictions can move the scores substantially. The result does not measure split-to-split variability or uncertainty from new populations or tuning. Tuned classical models receive many more dataset-specific fitting attempts than Kumo, so compute budgets are not matched. Timing columns are diagnostic only: model loading, caching, libraries and hardware affect them, so they should not be read as a controlled speed comparison. Air Quality is randomly split across days; this is not a future-time forecasting evaluation. Results may vary with hardware and software versions.

Requires **Python 3.12+** and internet access (Air Quality and Auto MPG CSVs; Kumo weights on first use). CUDA is used when available, otherwise CPU. Lower `N_TRIALS` for a quicker, less thoroughly tuned run. Principal package versions are pinned in the install cell.

Sources: [model card](https://huggingface.co/nvidia/Kumo-Tabular), [Kumo API](https://nvidia.github.io/structured-data-models/api/generated/sdm.models.KumoTabular.html), [Air Quality](https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/airquality.html), [Auto MPG](https://vincentarelbundock.github.io/Rdatasets/doc/ISLR/Auto.html). Air Quality uses complete cases (111 of 153 rows). Both CSVs are loaded from a pinned [Rdatasets](https://github.com/vincentarelbundock/Rdatasets) revision. Kumo weights: OpenMDW 1.1.


In [ ]:
%pip install structured-data-models==0.2.1 scikit-learn==1.9.1 xgboost==3.4.1 catboost==1.2.10 optuna==5.0.0 pandas==3.1.0


In [1]:
from time import perf_counter

import numpy as np
import optuna
import pandas as pd
import sdm
import torch
from catboost import CatBoostClassifier, CatBoostRegressor
from IPython.display import display
from sklearn.datasets import load_breast_cancer, load_diabetes, load_iris, load_wine
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (
    GradientBoostingClassifier, GradientBoostingRegressor,
    HistGradientBoostingClassifier, HistGradientBoostingRegressor,
)
from sklearn.metrics import (
    accuracy_score, f1_score, log_loss,
    mean_absolute_error, mean_squared_error, r2_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier, XGBRegressor

SEED, SIZE, NUM_ESTIMATORS = 42, "small", 8
SPLIT_SEEDS = [42]
N_TRIALS, CV_FOLDS = 5, 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"Device: {device}; Kumo size: {SIZE}; split seeds: {SPLIT_SEEDS}; Optuna trials: {N_TRIALS}; CV folds: {CV_FOLDS}")


/Users/maciejnasinski/kumo/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu; Kumo size: small; split seeds: [42]; Optuna trials: 5; CV folds: 5


In [2]:
def engineer_iris(X):
    return X.assign(
        petal_area=X["petal length (cm)"] * X["petal width (cm)"],
        sepal_area=X["sepal length (cm)"] * X["sepal width (cm)"],
        petal_aspect=X["petal length (cm)"] / X["petal width (cm)"],
        petal_to_sepal_length=X["petal length (cm)"] / X["sepal length (cm)"],
    )


def engineer_wine(X):
    return X.assign(
        flavanoid_share=X["flavanoids"] / X["total_phenols"],
        color_to_hue=X["color_intensity"] / X["hue"],
        log_proline=np.log(X["proline"]),
        flavanoids_x_od=X["flavanoids"] * X["od280/od315_of_diluted_wines"],
    )


def engineer_breast_cancer(X):
    eps = 1e-6  # "mean concave points" can be 0.
    return X.assign(
        radius_growth=X["worst radius"] / X["mean radius"],
        texture_growth=X["worst texture"] / X["mean texture"],
        concave_points_growth=(X["worst concave points"] + eps) / (X["mean concave points"] + eps),
        shape_irregularity=X["mean perimeter"] ** 2 / X["mean area"],
        worst_area_x_concave_points=X["worst area"] * X["worst concave points"],
    )


def engineer_diabetes(X):
    return X.assign(
        bmi_x_bp=X["bmi"] * X["bp"],
        bmi_x_s5=X["bmi"] * X["s5"],
        bp_x_s5=X["bp"] * X["s5"],
        ldl_to_hdl=X["s2"] / X["s3"],
        bmi_squared=X["bmi"] ** 2,
    )


def engineer_air_quality(X):
    dates = pd.to_datetime(pd.DataFrame({"year": 1973, "month": X["Month"], "day": X["Day"]}))
    return X.assign(
        day_of_year=dates.dt.day_of_year,
        temp_per_wind=X["Temp"] / X["Wind"],
        solar_x_temp=X["Solar.R"] * X["Temp"],
        inverse_wind=1 / X["Wind"],
    )


def engineer_auto_mpg(X):
    return X.assign(
        power_to_weight=X["horsepower"] / X["weight"],
        displacement_per_cylinder=X["displacement"] / X["cylinders"],
        log_weight=np.log(X["weight"]),
        origin_usa=(X["origin"] == 1).astype(int),
        origin_europe=(X["origin"] == 2).astype(int),
        origin_japan=(X["origin"] == 3).astype(int),
    ).drop(columns="origin")


rdatasets = (
    "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/"
    "1dcc2bf5f955cc1224a3e1307256e1fe86b68dae/csv"
)
air = pd.read_csv(
    f"{rdatasets}/datasets/airquality.csv",
    usecols=["Ozone", "Solar.R", "Wind", "Temp", "Month", "Day"],
).dropna()
auto = pd.read_csv(
    f"{rdatasets}/ISLR/Auto.csv",
    usecols=["mpg", "cylinders", "displacement", "horsepower", "weight",
             "acceleration", "year", "origin"],
).dropna()

iris, wine = load_iris(as_frame=True), load_wine(as_frame=True)
cancer, diabetes = load_breast_cancer(as_frame=True), load_diabetes(as_frame=True, scaled=False)
DATASETS = {  # name: (task, features, target, feature engineering)
    "Iris": ("classification", iris.data, iris.target, engineer_iris),
    "Wine": ("classification", wine.data, wine.target, engineer_wine),
    "Breast Cancer": ("classification", cancer.data, cancer.target, engineer_breast_cancer),
    "Diabetes": ("regression", diabetes.data, diabetes.target, engineer_diabetes),
    "Air Quality": ("regression", air.drop(columns="Ozone"), air["Ozone"], engineer_air_quality),
    "Auto MPG": ("regression", auto.drop(columns="mpg"), auto["mpg"], engineer_auto_mpg),
}
for name, (task, X, y, engineer) in DATASETS.items():
    engineered = engineer(X)
    assert np.isfinite(engineered.to_numpy(dtype=float)).all(), name
    print(f"{name:14} {task:15} rows={len(X):4}  raw features={X.shape[1]:2}  engineered={engineered.shape[1]:2}")


Iris           classification  rows= 150  raw features= 4  engineered= 8
Wine           classification  rows= 178  raw features=13  engineered=17
Breast Cancer  classification  rows= 569  raw features=30  engineered=35
Diabetes       regression      rows= 442  raw features=10  engineered=15
Air Quality    regression      rows= 111  raw features= 5  engineered= 9
Auto MPG       regression      rows= 392  raw features= 7  engineered=12


In [3]:
MODELS = {
    "classification": ["Gradient Boosting", "Histogram Gradient Boosting", "XGBoost", "CatBoost", "LDA"],
    "regression": ["Gradient Boosting", "Histogram Gradient Boosting", "XGBoost", "CatBoost"],
}

SEARCH_SPACES = {
    "Gradient Boosting": lambda t: {
        "n_estimators": t.suggest_int("n_estimators", 50, 600, log=True),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": t.suggest_int("max_depth", 2, 5),
        "subsample": t.suggest_float("subsample", 0.6, 1.0),
        "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 20),
        "max_features": t.suggest_float("max_features", 0.5, 1.0),
    },
    "Histogram Gradient Boosting": lambda t: {
        "max_iter": t.suggest_int("max_iter", 50, 600, log=True),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_leaf_nodes": t.suggest_int("max_leaf_nodes", 4, 63, log=True),
        "min_samples_leaf": t.suggest_int("min_samples_leaf", 2, 40, log=True),
        "l2_regularization": t.suggest_float("l2_regularization", 1e-6, 10.0, log=True),
        "max_features": t.suggest_float("max_features", 0.5, 1.0),
    },
    "XGBoost": lambda t: {
        "n_estimators": t.suggest_int("n_estimators", 50, 1000, log=True),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": t.suggest_int("max_depth", 2, 8),
        "min_child_weight": t.suggest_float("min_child_weight", 0.5, 10.0, log=True),
        "subsample": t.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda": t.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "reg_alpha": t.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
    },
    "CatBoost": lambda t: {
        "iterations": t.suggest_int("iterations", 100, 1000, log=True),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": t.suggest_int("depth", 2, 8),
        "l2_leaf_reg": t.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        "random_strength": t.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": t.suggest_float("bagging_temperature", 0.0, 1.0),
    },
    # shrinkage=0 is classical LDA; larger values regularize the covariance estimate.
    "LDA": lambda t: {"solver": "lsqr", "shrinkage": t.suggest_float("shrinkage", 0.0, 1.0)},
}


def make_model(name, task, params):
    classification = task == "classification"
    if name == "Gradient Boosting":
        cls = GradientBoostingClassifier if classification else GradientBoostingRegressor
        return cls(random_state=SEED, **params)
    if name == "Histogram Gradient Boosting":
        cls = HistGradientBoostingClassifier if classification else HistGradientBoostingRegressor
        return cls(random_state=SEED, **params)
    if name == "XGBoost":
        cls = XGBClassifier if classification else XGBRegressor
        return cls(random_state=SEED, tree_method="hist", n_jobs=1, **params)
    if name == "CatBoost":
        cls = CatBoostClassifier if classification else CatBoostRegressor
        return cls(random_seed=SEED, thread_count=1, verbose=0, allow_writing_files=False, **params)
    if name == "LDA":
        return make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(**params))
    raise ValueError(f"Unknown model: {name}")


def tune(name, task, X, y):
    classification = task == "classification"
    cv = (StratifiedKFold if classification else KFold)(CV_FOLDS, shuffle=True, random_state=SEED)
    scoring = "neg_log_loss" if classification else "neg_root_mean_squared_error"

    def objective(trial):
        model = make_model(name, task, SEARCH_SPACES[name](trial))
        return cross_val_score(model, X, y, cv=cv, scoring=scoring, n_jobs=CV_FOLDS).mean()

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS)
    best = SEARCH_SPACES[name](optuna.trial.FixedTrial(study.best_params))
    return make_model(name, task, best), -study.best_value, best


KUMO = {}


def kumo_predict(task, X_train, y_train, X_test):
    classification = task == "classification"
    if task not in KUMO:
        started = perf_counter()
        KUMO[task] = sdm.models.KumoTabular(task=task, size=SIZE, device=device)
        print(f"Loaded Kumo {task} model in {perf_counter() - started:.1f}s")
    # Only training labels are included; test targets are never passed to Kumo.
    context = X_train.reset_index(drop=True).assign(target=y_train.to_numpy())
    stypes = sdm.infer_stypes(
        context, overrides={"target": "categorical" if classification else "numerical"},
    )
    train = sdm.TableTensor.from_pandas(context, stypes=stypes, device=device)
    query = sdm.TableTensor.from_pandas(
        X_test.reset_index(drop=True),
        stypes={column: stypes[column] for column in X_test.columns}, device=device,
    )
    with torch.inference_mode(), torch.amp.autocast(
        device.type, dtype=torch.float16, enabled=device.type == "cuda",
    ):
        output = KUMO[task](
            x_context=train.drop_columns("target"), y_context=train[:, "target"],
            x_query=query, num_estimators=NUM_ESTIMATORS,
            generator=torch.Generator(device=device).manual_seed(SEED),
        ).to_pandas()
    if not classification:
        return output["q500"].to_numpy(), None, None
    classes = output.columns.to_numpy().astype(int)
    probabilities = output.to_numpy()
    return classes[probabilities.argmax(axis=1)], probabilities, classes


def score(task, y_true, predictions, probabilities=None, classes=None):
    if task == "regression":
        return {
            "RMSE": np.sqrt(mean_squared_error(y_true, predictions)),
            "MAE": mean_absolute_error(y_true, predictions),
            "R2": r2_score(y_true, predictions),
        }
    order = np.argsort(classes)
    probabilities = probabilities[:, order]
    assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)
    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Macro F1": f1_score(y_true, predictions, average="macro"),
        "Log loss": log_loss(y_true, probabilities, labels=classes[order]),
    }


In [4]:
rows, best_params = [], {}
for split_seed in SPLIT_SEEDS:
    SEED = split_seed
    torch.manual_seed(SEED)
    print(f"Starting outer split {split_seed}")
    for dataset, (task, X, y, engineer) in DATASETS.items():
        classification = task == "classification"
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=SEED, stratify=y if classification else None,
        )
        for scenario in ["Raw", "Feature engineered"]:
            if scenario == "Feature engineered":
                X_train_s, X_test_s = engineer(X_train), engineer(X_test)
            else:
                X_train_s, X_test_s = X_train, X_test
            print(f"  {dataset} / {scenario}: {len(X_train_s)} train, {len(X_test_s)} test, {X_train_s.shape[1]} features")

            started = perf_counter()
            predictions, probabilities, classes = kumo_predict(task, X_train_s, y_train, X_test_s)
            if device.type == "cuda":
                torch.cuda.synchronize()
            rows.append({
                "Split seed": split_seed, "Task": task, "Dataset": dataset,
                "Scenario": scenario, "Model": f"Kumo ({SIZE})",
                **score(task, y_test, predictions, probabilities, classes),
                "CV objective": np.nan, "Tune (s)": 0.0,
                "Fit + predict (s)": perf_counter() - started,
            })

            for name in MODELS[task]:
                started = perf_counter()
                model, cv_objective, params = tune(name, task, X_train_s, y_train)
                tune_seconds = perf_counter() - started
                started = perf_counter()
                model.fit(X_train_s, y_train)
                predictions = np.asarray(model.predict(X_test_s)).ravel()
                probabilities = model.predict_proba(X_test_s) if classification else None
                if device.type == "cuda":
                    torch.cuda.synchronize()
                fit_seconds = perf_counter() - started
                assert predictions.shape == y_test.shape and np.isfinite(predictions).all()
                best_params[(split_seed, dataset, scenario, name)] = params
                rows.append({
                    "Split seed": split_seed, "Task": task, "Dataset": dataset,
                    "Scenario": scenario, "Model": name,
                    **score(task, y_test, predictions, probabilities,
                            model.classes_ if classification else None),
                    "CV objective": cv_objective, "Tune (s)": tune_seconds,
                    "Fit + predict (s)": fit_seconds,
                })

results = pd.DataFrame(rows)
assert len(results) == len(SPLIT_SEEDS) * 66
split_label = "split" if len(SPLIT_SEEDS) == 1 else "splits"
print(f"Done: {len(results)} model evaluations across {len(SPLIT_SEEDS)} {split_label}")


Starting outer split 42
  Iris / Raw: 112 train, 38 test, 4 features


Loaded Kumo classification model in 0.8s


  Iris / Feature engineered: 112 train, 38 test, 8 features


  Wine / Raw: 133 train, 45 test, 13 features


  Wine / Feature engineered: 133 train, 45 test, 17 features


  Breast Cancer / Raw: 426 train, 143 test, 30 features


... (4 outputs left out)

  Air Quality / Feature engineered: 83 train, 28 test, 9 features


  Auto MPG / Raw: 294 train, 98 test, 7 features


  Auto MPG / Feature engineered: 294 train, 98 test, 12 features


Done: 66 model evaluations across 1 split


## Classification results

Higher accuracy and macro F1 are better; lower log loss is better. `CV objective` is the best five-fold training CV log loss for tuned models. Timings are diagnostics, not controlled latency comparisons.


In [5]:
classification_results = (
    results.query("Task == 'classification'")
    .drop(columns=["Task", "Split seed", "RMSE", "MAE", "R2"])
    .set_index(["Dataset", "Scenario", "Model"])
    .sort_values(["Dataset", "Scenario", "Log loss"], ascending=[True, False, True])
)
display(classification_results.round(4))


Accuracy  \
Dataset       Scenario           Model                                   
Breast Cancer Raw                Kumo (small)                   0.9790   
                                 XGBoost                        0.9720   
                                 CatBoost                       0.9650   
                                 Gradient Boosting              0.9650   
                                 LDA                            0.9580   
                                 Histogram Gradient Boosting    0.9650   
              Feature engineered Kumo (small)                   0.9650   
                                 LDA                            0.9650   
                                 XGBoost                        0.9720   
                                 Gradient Boosting              0.9580   
                                 CatBoost                       0.9510   
                                 Histogram Gradient Boosting    0.9580   
Iris          Raw                Kumo (small)                   1.0000   
                                 LDA                            0.9211   
                                 CatBoost                       0.9474   
                                 Gradient Boosting              0.8947   
                                 XGBoost                        0.8947   
                                 Histogram Gradient Boosting    0.9474   
              Feature engineered LDA                            0.9737   
                                 Kumo (small)                   0.9211   
                                 XGBoost                        0.9211   
                                 Gradient Boosting              0.9211   
                                 CatBoost                       0.9211   
                                 Histogram Gradient Boosting    0.9211   
Wine          Raw                Histogram Gradient Boosting    1.0000   
                                 LDA                            1.0000   
                                 Gradient Boosting              1.0000   
                                 Kumo (small)                   1.0000   
                                 CatBoost                       1.0000   
                                 XGBoost                        1.0000   
              Feature engineered Histogram Gradient Boosting    1.0000   
                                 Gradient Boosting              1.0000   
                                 Kumo (small)                   1.0000   
                                 CatBoost                       1.0000   
                                 LDA                            0.9556   
                                 XGBoost                        1.0000   

                                                              Macro F1  \
Dataset       Scenario           Model                                   
Breast Cancer Raw                Kumo (small)                   0.9776   
                                 XGBoost                        0.9698   
                                 CatBoost                       0.9621   
                                 Gradient Boosting              0.9621   
                                 LDA                            0.9543   
                                 Histogram Gradient Boosting    0.9621   
              Feature engineered Kumo (small)                   0.9627   
                                 LDA                            0.9617   
                                 XGBoost                        0.9695   
                                 Gradient Boosting              0.9543   
                                 CatBoost                       0.9464   
                                 Histogram Gradient Boosting    0.9543   
Iris          Raw                Kumo (small)                   1.0000   
                                 LDA                            0.9230   
                                 CatBoost                       0

## Regression results

Lower RMSE and MAE are better; R² is higher-is-better. Kumo's `q500` point prediction is a median, which is naturally aligned with absolute error; report both MAE and RMSE when interpreting it. `CV objective` is the best five-fold training CV RMSE.


In [6]:
regression_results = (
    results.query("Task == 'regression'")
    .drop(columns=["Task", "Split seed", "Accuracy", "Macro F1", "Log loss"])
    .set_index(["Dataset", "Scenario", "Model"])
    .sort_values(["Dataset", "Scenario", "RMSE"], ascending=[True, False, True])
)
display(regression_results.round(4))


CV objective  \
Dataset     Scenario           Model                                       
Air Quality Raw                Kumo (small)                          NaN   
                               XGBoost                           16.4274   
                               Gradient Boosting                 15.2654   
                               Histogram Gradient Boosting       16.4586   
                               CatBoost                          16.4027   
            Feature engineered Kumo (small)                          NaN   
                               XGBoost                           15.3164   
                               CatBoost                          14.5701   
                               Gradient Boosting                 14.8104   
                               Histogram Gradient Boosting       15.6381   
Auto MPG    Raw                Kumo (small)                          NaN   
                               CatBoost                           2.8563   
                               Histogram Gradient Boosting        2.9932   
                               Gradient Boosting                  2.9344   
                               XGBoost                            3.0009   
            Feature engineered Kumo (small)                          NaN   
                               Histogram Gradient Boosting        3.0193   
                               CatBoost                           2.9114   
                               Gradient Boosting                  2.9502   
                               XGBoost                            3.0647   
Diabetes    Raw                Kumo (small)                          NaN   
                               Gradient Boosting                 58.8049   
                               CatBoost                          59.9195   
                               Histogram Gradient Boosting       62.1836   
                               XGBoost                           62.2294   
            Feature engineered Gradient Boosting                 59.2654   
                               CatBoost                          60.2909   
                               Kumo (small)                          NaN   
                               Histogram Gradient Boosting       61.8474   
                               XGBoost                           62.0369   

                                                            Tune (s)  \
Dataset     Scenario           Model                                   
Air Quality Raw                Kumo (small)                   0.0000   
                               XGBoost                        0.1420   
                               Gradient Boosting              0.2380   
                               Histogram Gradient Boosting    0.5588   
                               CatBoost                       0.1990   
            Feature engineered Kumo (small)                   0.0000   
                               XGBoost                        0.1787   
                               CatBoost                       0.3260   
                               Gradient Boosting              0.2627   
                               Histogram Gradient Boosting    0.5808   
Auto MPG    Raw                Kumo (small)                   0.0000   
                               CatBoost                       0.3798   
                               Histogram Gradient Boosting    0.6857   
                               Gradient Boosting              0.2821   
                               XGBoost                        0.1730   
            Feature engineered Kumo (small)                   0.0000   
                               Histogram Gradient Boosting    0.8128   
                               CatBoost                       0.6458   
                               Gradient Boosting              0.3258   
                               XGBoost                        0.1874   
Diabetes    Raw                Kumo (small)

## Summary

The primary metric is log loss for classification and RMSE for regression; lower is better. Feature-engineering changes compare the same test split.


In [7]:
results["Primary metric"] = results["Log loss"].fillna(results["RMSE"])
summary = results.pivot_table(
    index=["Task", "Dataset", "Model"], columns="Scenario", values="Primary metric",
)[["Raw", "Feature engineered"]]
summary["FE change"] = summary["Feature engineered"] - summary["Raw"]
display(summary.round(4))

results["Rank"] = results.groupby(["Dataset", "Scenario"])["Primary metric"].rank(method="min")
average_rank = results.groupby(["Task", "Model", "Scenario"])["Rank"].mean().unstack("Scenario")
display(average_rank.round(2))

winners = results.loc[results.groupby(["Dataset", "Scenario"])["Primary metric"].idxmin()]
display(winners.set_index(["Task", "Dataset", "Scenario"])[["Model", "Primary metric"]].sort_index().round(4))


Scenario                                                      Raw  \
Task           Dataset       Model                                  
classification Breast Cancer CatBoost                      0.0829   
                             Gradient Boosting             0.0865   
                             Histogram Gradient Boosting   0.1091   
                             Kumo (small)                  0.0620   
                             LDA                           0.0922   
                             XGBoost                       0.0805   
               Iris          CatBoost                      0.1737   
                             Gradient Boosting             0.2184   
                             Histogram Gradient Boosting   0.2907   
                             Kumo (small)                  0.0571   
                             LDA                           0.0933   
                             XGBoost                       0.2370   
               Wine          CatBoost                      0.0300   
                             Gradient Boosting             0.0152   
                             Histogram Gradient Boosting   0.0030   
                             Kumo (small)                  0.0213   
                             LDA                           0.0112   
                             XGBoost                       0.0460   
regression     Air Quality   CatBoost                     28.0450   
                             Gradient Boosting            26.5842   
                             Histogram Gradient Boosting  28.0366   
                             Kumo (small)                 24.1702   
                             XGBoost                      26.4480   
               Auto MPG      CatBoost                      2.3976   
                             Gradient Boosting             2.5028   
                             Histogram Gradient Boosting   2.4500   
                             Kumo (small)                  2.1325   
                             XGBoost                       2.5510   
               Diabetes      CatBoost                     53.3958   
                             Gradient Boosting            52.5789   
                             Histogram Gradient Boosting  55.0596   
                             Kumo (small)                 51.7668   
                             XGBoost                      56.3214   

Scenario                                                  Feature engineered  \
Task           Dataset       Model                                             
classification Breast Cancer CatBoost                                 0.0908   
                             Gradient Boosting                        0.0897   
                             Histogram Gradient Boosting              0.1034   
                             Kumo (small)                             0.0586   
                             LDA                                      0.0674   
                             XGBoost                                  0.0746   
               Iris          CatBoost                                 0.1471   
                             Gradient Boosting                        0.1431   
                             Histogram Gradient Boosting              0.2853   
                             Kumo (small)                             0.0880   
                             LDA                                      0.0838   
                             XGBoost                                  0.1409   
               Wine          CatBoost                                 0.0215   
                             Gradient Boosting                        0.0090   
                             Histogram Gradient Boosting              0.0032   
                             Kumo (small)                             0.0191   
                             LDA                                      0.0388   
                             XGBoost                            

Scenario                                    Feature engineered   Raw
Task           Model                                                
classification CatBoost                                   4.67  3.67
               Gradient Boosting                          3.33  3.67
               Histogram Gradient Boosting                4.33  4.33
               Kumo (small)                               2.00  2.00
               LDA                                        2.67  3.00
               XGBoost                                    4.00  4.33
regression     CatBoost                                   2.67  3.33
               Gradient Boosting                          3.00  3.00
               Histogram Gradient Boosting                3.67  3.67
               Kumo (small)                               1.67  1.00
               XGBoost                                    4.00  4.00

Model  \
Task           Dataset       Scenario                                          
classification Breast Cancer Feature engineered                 Kumo (small)   
                             Raw                                Kumo (small)   
               Iris          Feature engineered                          LDA   
                             Raw                                Kumo (small)   
               Wine          Feature engineered  Histogram Gradient Boosting   
                             Raw                 Histogram Gradient Boosting   
regression     Air Quality   Feature engineered                 Kumo (small)   
                             Raw                                Kumo (small)   
               Auto MPG      Feature engineered                 Kumo (small)   
                             Raw                                Kumo (small)   
               Diabetes      Feature engineered            Gradient Boosting   
                             Raw                                Kumo (small)   

                                                 Primary metric  
Task           Dataset       Scenario                            
classification Breast Cancer Feature engineered          0.0586  
                             Raw                         0.0620  
               Iris          Feature engineered          0.0838  
                             Raw                         0.0571  
               Wine          Feature engineered          0.0032  
                             Raw                         0.0030  
regression     Air Quality   Feature engineered         22.8785  
                             Raw                        24.1702  
               Auto MPG      Feature engineered          2.1855  
                             Raw                         2.1325  
               Diabetes      Feature engineered         51.1700  
                             Raw                        51.7668

## Tuned hyperparameters

Best configuration found by Optuna for each outer split, dataset, scenario, and tuned model. LDA's scaler is fit inside each CV fold and refit with the selected model on the full outer training split.


In [8]:
with pd.option_context("display.max_colwidth", None):
    display(pd.Series(
        {key: {k: round(v, 4) if isinstance(v, float) else v for k, v in params.items()}
         for key, params in best_params.items()},
        name="Best parameters",
    ).rename_axis(["Split seed", "Dataset", "Scenario", "Model"]).to_frame())


Best parameters
Split seed Dataset       Scenario           Model                                                                                                                                                                                                           
42         Iris          Raw                Gradient Boosting                                                             {'n_estimators': 145, 'learning_rate': 0.0269, 'max_depth': 4, 'subsample': 0.6558, 'min_samples_leaf': 6, 'max_features': 0.6832}
                                            Histogram Gradient Boosting                                         {'max_iter': 145, 'learning_rate': 0.0269, 'max_leaf_nodes': 21, 'min_samples_leaf': 2, 'l2_regularization': 0.0001, 'max_features': 0.6832}
                                            XGBoost                      {'n_estimators': 124, 'learning_rate': 0.0596, 'max_depth': 5, 'min_child_weight': 1.1964, 'subsample': 0.8447, 'colsample_bytree': 0.5697, 'reg_lambda': 0.0147, 'reg_alpha': 0.0}
                                            CatBoost                                                               {'iterations': 114, 'learning_rate': 0.1903, 'depth': 6, 'l2_leaf_reg': 5.1059, 'random_strength': 0.0012, 'bagging_temperature': 0.9699}
                                            LDA                                                                                                                                                                       {'solver': 'lsqr', 'shrinkage': 0.156}
                         Feature engineered Gradient Boosting                                                             {'n_estimators': 145, 'learning_rate': 0.0269, 'max_depth': 4, 'subsample': 0.6558, 'min_samples_leaf': 6, 'max_features': 0.6832}
                                            Histogram Gradient Boosting                                         {'max_iter': 145, 'learning_rate': 0.0269, 'max_leaf_nodes': 21, 'min_samples_leaf': 2, 'l2_regularization': 0.0001, 'max_features': 0.6832}
                                            XGBoost                       {'n_estimators': 195, 'learning_rate': 0.1445, 'max_depth': 3, 'min_child_weight': 2.3335, 'subsample': 0.837, 'colsample_bytree': 0.5232, 'reg_lambda': 0.2693, 'reg_alpha': 0.0}
                                            CatBoost                                                               {'iterations': 270, 'learning_rate': 0.0269, 'depth': 6, 'l2_leaf_reg': 1.3788, 'random_strength': 0.0147, 'bagging_temperature': 0.3664}
                                            LDA                                                                                                                                                                      {'solver': 'lsqr', 'shrinkage': 0.3745}
           Wine          Raw                Gradient Boosting                                                             {'n_estimators': 395, 'learning_rate': 0.0206, 'max_depth': 2, 'subsample': 0.6734, 'min_samples_leaf': 7, 'max_features': 0.7624}
                                            Histogram Gradient Boosting                                           {'max_iter': 155, 'learning_rate': 0.1445, 'max_leaf_nodes': 6, 'min_samples_leaf': 8, 'l2_regularization': 0.014, 'max_features': 0.5232}
                                            XGBoost                      {'n_estimators': 124, 'learning_rate': 0.0596, 'max_depth': 5, 'min_child_weight': 1.1964, 'subsample': 0.8447, 'colsample_bytree': 0.5697, 'reg_lambda': 0.0147, 'reg_alpha': 0.0}
                                            CatBoost                                                               {'iterations': 285, 'learning_rate': 0.1445, 'depth': 3, 'l2_leaf_reg': 3.2676, 'random_strength': 0.2342, 'bagging_temperature': 0.0465}
                                            LDA                                                                                                                                              